# BİL458 Bilgisayarlı Görme - Ödev 1
## Laboratuvar Görevleri L1.1 - L1.4

**Ad Soyad:** Sevban Bozaslan

**Öğrenci No:** 23120205020

---

Bu ödevde dijital kamera işlem hattının dört ayrı adımındaki bilgi kaybı sayısal olarak ölçülmüştür. Her görev, ders slaytlarında verilen bir iddianın deneysel doğrulamasıdır.

| Görev | İşlem hattı adımı | Ölçülen büyüklük |
|---|---|---|
| L1.1 | Renk süzgeç dizisi (Bayer CFA) | PSNR, kanal bazlı hata |
| L1.2 | Analog-sayısal çevrim (ADC) | SNR, bit başına kazanç |
| L1.3 | Gamma kodlaması | Ortalama alma hatası (ton) |
| L1.4 | Mercek geometrisi | GSD, şerit genişliği |

In [ ]:
import urllib.request
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

OUT = Path("out")
OUT.mkdir(exist_ok=True)

IMG_PATH = Path("kodim19.png")
if not IMG_PATH.exists():
    urllib.request.urlretrieve("https://r0k.us/graphics/kodak/kodak/kodim19.png", IMG_PATH)

print("numpy  :", np.__version__)
print("opencv :", cv2.__version__)
print("test goruntusu hazir:", IMG_PATH.exists())

---

# L1.1 - Bayer deseni ve demozaikleme kaybı

## Problem

Bir görüntü sensöründeki her fotodiyot renk körüdür; yalnızca kendisine ulaşan foton sayısını ölçer. Renk bilgisi, her fotositin üzerine yerleştirilen renk süzgeciyle elde edilir. Bunun bedeli, her pikselde üç kanaldan yalnızca birinin ölçülmesidir.

En yaygın süzgeç dizilimi **Bayer RGGB** desenidir ve 2x2 blok halinde tekrar eder:

```
R G R G
G B G B
R G R G
G B G B
```

Piksellerin dörtte biri R, dörtte biri B, yarısı G ölçer. Yeşilin iki kat sık örneklenmesinin nedeni, parlaklık (luminance) sinyalinin ağırlıklı olarak yeşil banttan gelmesi ve insan görme sisteminin parlaklıktaki yüksek frekanslı detaya, renkteki aynı detaydan belirgin biçimde daha duyarlı olmasıdır.

Eksik iki kanalın komşu piksellerden kestirilmesi işlemine **demozaikleme** denir. Bu bir ölçüm değil tahmindir; ödevin amacı bu tahminin hatasını nicel olarak ölçmektir.

In [ ]:
img = np.array(plt.imread(IMG_PATH), dtype=np.float64)
if img.max() <= 1.0:
    img = img * 255.0
img = img[:, :, :3]
H, W, _ = img.shape

print("boyut :", img.shape)
print("aralik: %.1f - %.1f" % (img.min(), img.max()))

plt.figure(figsize=(5, 7.5))
plt.imshow(img.astype(np.uint8))
plt.axis("off")
plt.title("Test goruntusu: Kodak kodim19")
plt.show()

## Adım 1: Mozaikleme

Tam renkli referans görüntüden her pikselin yalnızca desendeki kanalı bırakılarak ham sensör verisi (RAW) üretilir. Maskeler satır ve sütun paritesine göre tanımlanır:

| satır | sütun | süzgeç |
|---|---|---|
| çift | çift | R |
| çift | tek | G |
| tek | çift | G |
| tek | tek | B |

In [ ]:
def bayer_masks(h, w):
    mr = np.zeros((h, w), dtype=bool)
    mg = np.zeros((h, w), dtype=bool)
    mb = np.zeros((h, w), dtype=bool)
    mr[0::2, 0::2] = True
    mg[0::2, 1::2] = True
    mg[1::2, 0::2] = True
    mb[1::2, 1::2] = True
    return mr, mg, mb


MR, MG, MB = bayer_masks(H, W)

raw = np.zeros((H, W), dtype=np.float64)
raw[MR] = img[:, :, 0][MR]
raw[MG] = img[:, :, 1][MG]
raw[MB] = img[:, :, 2][MB]

print("R piksel: %7d  (%.1f%%)" % (MR.sum(), 100 * MR.sum() / MR.size))
print("G piksel: %7d  (%.1f%%)" % (MG.sum(), 100 * MG.sum() / MG.size))
print("B piksel: %7d  (%.1f%%)" % (MB.sum(), 100 * MB.sum() / MB.size))
print()
print("orijinal olcum sayisi   : %d" % img.size)
print("ham sensor olcum sayisi : %d" % raw.size)
print("kaybedilen bilgi        : %.1f%%" % (100 * (1 - raw.size / img.size)))

In [ ]:
cfa_rgb = np.zeros_like(img)
cfa_rgb[:, :, 0][MR] = raw[MR]
cfa_rgb[:, :, 1][MG] = raw[MG]
cfa_rgb[:, :, 2][MB] = raw[MB]

fig, ax = plt.subplots(1, 3, figsize=(13, 5))
ax[0].imshow(raw, cmap="gray", vmin=0, vmax=255)
ax[0].set_title("Ham sensor verisi (RAW, tek kanal)")
ax[1].imshow(raw[200:216, 200:216], cmap="gray", vmin=0, vmax=255, interpolation="nearest")
ax[1].set_title("16x16 buyutulmus RAW")
ax[2].imshow(cfa_rgb[200:216, 200:216].astype(np.uint8), interpolation="nearest")
ax[2].set_title("Ayni bolge, suzgec renkleriyle")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.savefig(OUT / "02_mozaik.png", dpi=120)
plt.show()

## Adım 2: Bilineer demozaikleme

Eksik değerler, o kanalın ölçüldüğü komşu piksellerin ağırlıklı ortalamasıyla kestirilir. Konum tipine göre komşuluk yapısı değişir:

| Bulunulan konum | Eksik kanal | Kullanılan komşular |
|---|---|---|
| R | G | 4 dik komşu |
| R | B | 4 köşegen komşu |
| B | G | 4 dik komşu |
| B | R | 4 köşegen komşu |
| G (çift satır) | R | 2 yatay komşu |
| G (çift satır) | B | 2 dikey komşu |
| G (tek satır) | R | 2 dikey komşu |
| G (tek satır) | B | 2 yatay komşu |

Bu sekiz durum, maskelenmiş seyrek dizilere uygulanan iki konvolüsyon çekirdeğiyle tek adımda karşılanır:

$$K_G = \frac{1}{4}\begin{bmatrix}0&1&0\\1&4&1\\0&1&0\end{bmatrix} \qquad K_{RB} = \frac{1}{4}\begin{bmatrix}1&2&1\\2&4&2\\1&2&1\end{bmatrix}$$

Seyrek dizide ölçüm yapılmayan konumlar sıfır olduğundan toplama katkı vermez; çekirdek her konumda kendiliğinden yalnızca doğru komşuları toplar. Ağırlıklar, her konum tipinde katkı veren terimlerin toplamı tam 4 olacak biçimde seçilmiştir. Bu nedenle tek bir `/4` bölmesi tüm durumlarda doğru ortalamayı verir ve ortalama parlaklık kaymaz.

**Not:** G kanalı her zaman dört komşudan kestirilirken, R ve B kanalları G konumlarında yalnızca iki komşudan ve tek yönden kestirilir. Kanallar arası PSNR farkının kaynağı budur.

In [ ]:
K_G = np.array([[0.0, 1.0, 0.0],
                [1.0, 4.0, 1.0],
                [0.0, 1.0, 0.0]]) / 4.0

K_RB = np.array([[1.0, 2.0, 1.0],
                 [2.0, 4.0, 2.0],
                 [1.0, 2.0, 1.0]]) / 4.0


def conv(x, k):
    return cv2.filter2D(x, -1, k, borderType=cv2.BORDER_REFLECT)


def demosaic_bilinear(raw, mr, mg, mb):
    out = np.zeros(raw.shape + (3,), dtype=np.float64)
    out[:, :, 0] = conv(raw * mr, K_RB)
    out[:, :, 1] = conv(raw * mg, K_G)
    out[:, :, 2] = conv(raw * mb, K_RB)
    return out


rec = demosaic_bilinear(raw, MR, MG, MB)
print("geri kurulmus goruntu:", rec.shape)

### Çekirdeklerin doğrulanması

Aşağıdaki 6x6 örnekte dört konum tipi için sonuçlar elle hesaplanabilir. Örneğin (2,2) konumu R süzgeçlidir; eksik G değeri dört dik komşunun ortalaması (160+165+135+155)/4 = 153.75, eksik B değeri dört köşegen komşunun ortalaması (60+70+55+75)/4 = 65 olmalıdır.

In [ ]:
demo_raw = np.array([
    [100, 140, 120, 150, 110, 145],
    [130,  60, 160,  70, 135,  65],
    [105, 135, 125, 155, 115, 140],
    [125,  55, 165,  75, 130,  60],
    [110, 145, 130, 160, 120, 150],
    [140,  65, 155,  80, 145,  70]], dtype=np.float64)

dh, dw = demo_raw.shape
dmr, dmg, dmb = bayer_masks(dh, dw)
demo_rec = demosaic_bilinear(demo_raw, dmr, dmg, dmb)

print("konum   suzgec        R         G         B")
for (y, x) in [(2, 2), (2, 3), (3, 2), (3, 3)]:
    tip = "R" if dmr[y, x] else ("G" if dmg[y, x] else "B")
    print("(%d,%d)     %s      %7.2f   %7.2f   %7.2f" % (
        y, x, tip, demo_rec[y, x, 0], demo_rec[y, x, 1], demo_rec[y, x, 2]))

## Adım 3: PSNR ile hata ölçümü

Geri kurulmuş görüntü referansla karşılaştırılır. Ortalama karesel hata ve tepe sinyal gürültü oranı:

$$MSE = \frac{1}{n}\sum_x \left[I(x) - \hat{I}(x)\right]^2 \qquad PSNR = 10\log_{10}\frac{I_{max}^2}{MSE}$$

8 bit görüntü için $I_{max} = 255$. Yüksek PSNR düşük hata anlamına gelir.

In [ ]:
def psnr(a, b, peak=255.0):
    mse = np.mean((a - b) ** 2)
    return float("inf") if mse == 0 else 10.0 * np.log10(peak ** 2 / mse)


print("kanal      PSNR         MSE")
for i, name in enumerate(["R", "G", "B"]):
    print("%s      %6.2f dB   %8.2f" % (
        name, psnr(img[:, :, i], rec[:, :, i]),
        np.mean((img[:, :, i] - rec[:, :, i]) ** 2)))
print("toplam %6.2f dB   %8.2f" % (psnr(img, rec), np.mean((img - rec) ** 2)))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 8))
ax[0].imshow(img.astype(np.uint8))
ax[0].set_title("Orijinal (referans)")
ax[1].imshow(np.clip(rec, 0, 255).astype(np.uint8))
ax[1].set_title("Demozaiklenmis (%.2f dB)" % psnr(img, rec))
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.savefig(OUT / "01_orijinal_vs_geri_kurulmus.png", dpi=120)
plt.show()

## Adım 4: Hata haritası ve hatanın konumsal dağılımı

Bilineer kestirim, komşu piksellerin birbirine benzediği varsayımına dayanır. Bu varsayım düz bölgelerde geçerlidir, kenarlarda çöker. Aşağıda hata, Sobel gradyan büyüklüğüne göre ayrılan iki piksel kümesinde karşılaştırılmıştır.

In [ ]:
err = np.abs(img - rec)

print("kanal   ortalama hata   maks hata")
for i, name in enumerate(["R", "G", "B"]):
    print("%s        %6.2f       %7.2f" % (name, err[:, :, i].mean(), err[:, :, i].max()))

gray = img.mean(axis=2)
gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
grad = np.hypot(gx, gy)

edge = grad >= np.percentile(grad, 90)
flat = grad <= np.percentile(grad, 50)
err_sum = err.sum(axis=2)

print()
print("kenar pikselleri (ust yuzde 10 gradyan): ortalama hata %6.2f" % err_sum[edge].mean())
print("duz pikseller    (alt yuzde 50 gradyan): ortalama hata %6.2f" % err_sum[flat].mean())
print("oran                                   : %.1f kat" % (err_sum[edge].mean() / err_sum[flat].mean()))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 7))
for i, name in enumerate(["R", "G", "B"]):
    im = ax[i].imshow(err[:, :, i], cmap="inferno", vmin=0, vmax=60)
    ax[i].set_title("%s kanali mutlak hata" % name)
    ax[i].axis("off")
fig.colorbar(im, ax=ax, fraction=0.03)
plt.savefig(OUT / "03_hata_haritasi.png", dpi=120, bbox_inches="tight")
plt.show()

## Adım 5: Zipper artefaktı

Keskin bir kenarda üç kanal farklı komşulardan ve kısmen farklı yönlerden kestirildiği için birbirinden bağımsız hata yapar. Sonuç, sahnede bulunmayan renk saçaklarının belirmesi ve kenar boyunca dişli bir desen oluşmasıdır.

Aşağıda hata yoğunluğu en yüksek 32x32 bölge otomatik olarak bulunmuş ve büyütülerek gösterilmiştir.

In [ ]:
box = cv2.boxFilter(err_sum, -1, (32, 32), normalize=False)
cy, cx = np.unravel_index(np.argmax(box), box.shape)
y0 = int(np.clip(cy - 16, 0, H - 32))
x0 = int(np.clip(cx - 16, 0, W - 32))
print("en bozuk 32x32 bolge: satir %d-%d, sutun %d-%d" % (y0, y0 + 32, x0, x0 + 32))

crop_o = img[y0:y0 + 32, x0:x0 + 32]
crop_r = rec[y0:y0 + 32, x0:x0 + 32]

fig, ax = plt.subplots(1, 3, figsize=(14, 5))
ax[0].imshow(crop_o.astype(np.uint8), interpolation="nearest")
ax[0].set_title("Orijinal")
ax[1].imshow(np.clip(crop_r, 0, 255).astype(np.uint8), interpolation="nearest")
ax[1].set_title("Demozaiklenmis (zipper)")
im = ax[2].imshow(np.abs(crop_o - crop_r).sum(axis=2), cmap="inferno", interpolation="nearest")
ax[2].set_title("Mutlak fark")
for a in ax:
    a.axis("off")
fig.colorbar(im, ax=ax[2], fraction=0.046)
plt.savefig(OUT / "04_zipper.png", dpi=150, bbox_inches="tight")
plt.show()

## L1.1 Sonuç

1. Ham sensör verisi, tam renkli görüntünün yalnızca üçte birini içerir; kalan üçte iki kestirimdir.
2. G kanalının PSNR değeri R ve B kanallarından yaklaşık 5 dB yüksektir. Neden, G kanalının iki kat sık örneklenmesi ve her zaman dört komşudan kestirilmesidir. R ve B kanalları G konumlarında yalnızca iki komşudan ve tek yönden kestirilir.
3. Hata düz bölgelerde ihmal edilebilir düzeydedir, kenarlarda yaklaşık altı kat artar. Bilineer kestirimin dayandığı komşu benzerliği varsayımı kenarlarda geçersizdir.
4. Zipper artefaktı, üç kanalın bağımsız kestirilmesinin doğrudan sonucudur. Örnekleme frekansına yakın desenlerde R ve B kanalları Nyquist sınırını aştığı için renk moiré deseni ortaya çıkar.

---

# L1.2 - Nicemleme gürültüsü ve SNR yasası

## Problem

Fotodiyot çıkışındaki sürekli sinyal, B bitlik bir ADC tarafından $2^B$ ayrık seviyeye yuvarlanır. Tam ölçek aralığı 1'e normalize edilirse adım büyüklüğü:

$$\Delta = \frac{1}{2^B}$$

Nicemleme hatası $[-\Delta/2, +\Delta/2]$ aralığında düzgün dağılımlıdır ve varyansı:

$$\sigma_e^2 = \frac{\Delta^2}{12}$$

B bir birim arttığında $\Delta$ yarıya, hata gücü dörtte bire iner. Buna karşılık gelen SNR kazancı $10\log_{10}4 = 6.02$ dB'dir. Bu görevde doğrusal bir rampa B = 2...8 bit ile nicemlenerek bu eğim deneysel olarak ölçülmüştür.

In [ ]:
N = 200000
ramp = np.linspace(0.0, 1.0, N, endpoint=False)


def quantize(x, bits):
    levels = 2 ** bits
    idx = np.clip(np.floor(x * levels), 0, levels - 1)
    return (idx + 0.5) / levels


def snr_db(sig, q):
    e = sig - q
    return 10.0 * np.log10(np.var(sig) / np.mean(e ** 2))


bits = np.arange(2, 9)
measured = np.array([snr_db(ramp, quantize(ramp, b)) for b in bits])
theory = 6.02 * bits

print(" B   seviye       delta      olculen SNR   teori 6.02B     fark")
for b, m, t in zip(bits, measured, theory):
    print("%2d   %6d   %.3e     %8.2f dB    %8.2f dB   %+6.2f" %
          (b, 2 ** b, 1 / 2 ** b, m, t, m - t))

In [ ]:
slope, intercept = np.polyfit(bits, measured, 1)
print("dogrusal uyum : SNR = %.3f * B + %.3f" % (slope, intercept))
print("olculen egim  : %.3f dB/bit" % slope)
print("teorik egim   : %.3f dB/bit   (20*log10(2))" % (20 * np.log10(2)))
print("bagil sapma   : %.2f%%" % (100 * abs(slope - 20 * np.log10(2)) / (20 * np.log10(2))))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot(bits, measured, "o-", label="olculen")
ax[0].plot(bits, theory, "--", label="teori: 6.02 B")
ax[0].set_xlabel("bit derinligi B")
ax[0].set_ylabel("SNR (dB)")
ax[0].set_title("SNR - B  (olculen egim = %.2f dB/bit)" % slope)
ax[0].grid(alpha=0.3)
ax[0].legend()

for b in [2, 3, 8]:
    ax[1].plot(ramp[::200], quantize(ramp, b)[::200], label="B=%d" % b)
ax[1].plot(ramp[::200], ramp[::200], "k--", lw=0.8, label="surekli")
ax[1].set_xlabel("giris")
ax[1].set_ylabel("nicemlenmis cikis")
ax[1].set_title("Nicemleme merdiveni")
ax[1].grid(alpha=0.3)
ax[1].legend()
plt.tight_layout()
plt.savefig(OUT / "05_snr_b.png", dpi=120)
plt.show()

In [ ]:
ramp2d = np.tile(np.linspace(0, 1, 512), (160, 1))
fig, ax = plt.subplots(4, 1, figsize=(11, 7))
for a, b in zip(ax, [2, 3, 4, 8]):
    a.imshow(quantize(ramp2d, b), cmap="gray", vmin=0, vmax=1, aspect="auto")
    a.set_ylabel("B=%d" % b)
    a.set_xticks([])
    a.set_yticks([])
ax[0].set_title("Bant olusumu (banding)")
plt.tight_layout()
plt.savefig(OUT / "06_banding.png", dpi=120)
plt.show()

## L1.2 Sonuç

1. Ölçülen eğim 6.021 dB/bit, teorik değer $20\log_{10}2 = 6.0206$ dB/bit ile birebir örtüşmektedir. Bağıl sapma sıfırdır.
2. Doğrusal uyumun sabit terimi 0.000 çıkmıştır; slaytta verilen $SNR = 6.02B + 1.76$ ifadesindeki 1.76 dB sabiti burada gözlenmemiştir. Bu bir tutarsızlık değildir: 1.76 dB sabiti ($10\log_{10}1.5$) girişin tam ölçekli sinüzoidal olduğu varsayımından türer. Bu görevdeki sinyal doğrusal rampadır ve değerleri tam ölçek aralığına düzgün dağıldığından sinyal varyansı $1/12$, gürültü varyansı $\Delta^2/12$ olur; oranın logaritması tam olarak $6.02B$ verir ve sabit terim sıfırlanır. **Eğim her iki sinyal tipinde de aynıdır**, yalnızca sabit terim sinyalin dalga biçimine bağlıdır.
3. Düşük bit derinliklerinde rampa üzerinde bant oluşumu (banding) gözlenmektedir. B = 2 için 4, B = 3 için 8 düz şerit ortaya çıkar. Yuvarlama, aynı seviyeye düşen tüm giriş değerlerini tek bir çıkışa indirger. Şeritler arasındaki süreksizlik, gözün düz alanlar arası küçük farkları abartarak algılaması nedeniyle olduğundan belirgin görünür. Bu, gradyan ağırlıklı içerikte 8 bitin yetersiz kalabilmesinin nedenidir.

---

# L1.3 - Gamma uzayında ortalama alma hatası

## Problem

Fiziksel ışık doğrusal toplanır; sensör de doğrusal ölçer. Buna karşılık insan görme sistemi Weber-Fechner yasası uyarınca yaklaşık logaritmik yanıt verir ve karanlık bölgedeki küçük farklara aydınlık bölgedekinden çok daha duyarlıdır.

Sınırlı bit bütçesini gözün duyarlı olduğu bölgeye ayırmak için kameralar kaydetmeden önce gamma kodlaması uygular:

$$I_{kodlu} = I_{dogrusal}^{1/\gamma}, \qquad \gamma \approx 2.2$$

Bunun sonucu olarak görüntü dosyalarındaki değerler ışık miktarı değildir. Ortalama alma doğrusal bir işlem, gamma kodlaması doğrusal olmayan bir dönüşüm olduğundan, kodlu değerler üzerinde doğrudan ortalama almak sistematik hata üretir. Doğru sıra: çöz, doğrusal uzayda ortala, yeniden kodla.

In [ ]:
GAMMA = 2.2


def decode(e):
    return e ** GAMMA


def encode(l):
    return l ** (1.0 / GAMMA)


a, b = 0.0, 1.0
naive = (a + b) / 2
correct = encode((decode(a) + decode(b)) / 2)

print("--- siyah + beyaz cifti ---")
print("naif  (kodlu uzayda ortalama) : %.4f  ->  8 bit %3d" % (naive, round(naive * 255)))
print("dogru (dogrusal uzayda)       : %.4f  ->  8 bit %3d" % (correct, round(correct * 255)))
print("sapma                         : %.4f  ->  %d ton" % (
    correct - naive, round(correct * 255) - round(naive * 255)))

## Bölge bazlı hata ölçümü

Parlaklığı soldan sağa artan bir gradyan görüntü üretilmiştir. Komşu piksel çiftleri kodlu uzayda sabit $\pm 0.10$ (8 bit ölçekte yaklaşık 26 ton) fark edecek biçimde ayarlanmış, ardından her çift yarı boyuta indirgenmiştir. Taban aralığı, hiçbir noktada kırpma (clipping) oluşmayacak şekilde seçilmiştir; böylece ölçülen fark yalnızca gamma etkisini yansıtır.

In [ ]:
W_G = 512
DELTA = 0.10

base_enc_row = np.linspace(DELTA, 1.0 - DELTA, W_G)
base_enc_img = np.tile(base_enc_row, (128, 1))
ripple = np.where(np.arange(W_G) % 2 == 0, DELTA, -DELTA)
enc = base_enc_img + ripple

naive_pair = (enc[:, 0::2] + enc[:, 1::2]) / 2.0
correct_pair = encode((decode(enc[:, 0::2]) + decode(enc[:, 1::2])) / 2.0)
err_tone = (correct_pair - naive_pair) * 255.0
base_enc = base_enc_img[:, 0::2]

print("isaret: pozitif deger, naif sonucun olmasi gerekenden KOYU oldugunu gosterir.")
print()
print("bolge    kodlu taban     ort. hata   maks hata   (8 bit ton)")
for lo, hi, name in [(0.0, 1 / 3, "koyu"), (1 / 3, 2 / 3, "orta"), (2 / 3, 1.0, "acik")]:
    m = (base_enc >= lo) & (base_enc < hi)
    print("%-8s %.2f - %.2f       %6.2f      %6.2f" % (
        name, lo, hi, err_tone[m].mean(), err_tone[m].max()))
print("%-8s %.2f - %.2f       %6.2f      %6.2f" % (
    "tumu", 0.0, 1.0, err_tone.mean(), err_tone.max()))
print()
print("koyu / acik hata orani: %.1f kat" % (
    err_tone[base_enc < 1 / 3].mean() / err_tone[base_enc >= 2 / 3].mean()))

## Örnek piksel hesabı

Koyu bölgeden alınan bir komşu piksel çifti için işlem adım adım gösterilmiştir.

In [ ]:
col = 60
e0, e1 = enc[0, col], enc[0, col + 1]
n = (e0 + e1) / 2
c = encode((decode(e0) + decode(e1)) / 2)

print("dosyadaki (kodlu, sRGB) degerler:")
print("  E0 = %.4f  (8 bit %3d)     E1 = %.4f  (8 bit %3d)" % (
    e0, round(e0 * 255), e1, round(e1 * 255)))
print()
print("adim 1 - coz (kodlu -> dogrusal isik):")
print("  L0 = %.4f^%.1f = %.5f" % (e0, GAMMA, decode(e0)))
print("  L1 = %.4f^%.1f = %.5f" % (e1, GAMMA, decode(e1)))
print("adim 2 - dogrusal uzayda ortala:")
print("  Lort = (%.5f + %.5f) / 2 = %.5f" % (
    decode(e0), decode(e1), (decode(e0) + decode(e1)) / 2))
print("adim 3 - yeniden kodla:")
print("  E = %.5f^(1/%.1f) = %.4f   ->  8 bit %d   [DOGRU]" % (
    (decode(e0) + decode(e1)) / 2, GAMMA, c, round(c * 255)))
print()
print("naif yol (kodlu uzayda dogrudan ortalama):")
print("  (%.4f + %.4f) / 2 = %.4f            ->  8 bit %d   [YANLIS]" % (
    e0, e1, n, round(n * 255)))
print()
print("fark: %.4f  ->  %.2f ton; naif sonuc daha koyu" % (c - n, (c - n) * 255))

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 8))
ax[0].imshow(np.clip(naive_pair, 0, 1), cmap="gray", vmin=0, vmax=1, aspect="auto")
ax[0].set_ylabel("naif")
ax[1].imshow(np.clip(correct_pair, 0, 1), cmap="gray", vmin=0, vmax=1, aspect="auto")
ax[1].set_ylabel("dogru")
im = ax[2].imshow(err_tone, cmap="inferno", aspect="auto")
ax[2].set_ylabel("hata (ton)")
for a_ in ax:
    a_.set_xticks([])
    a_.set_yticks([])
ax[0].set_title("Yari boyuta indirme: naif ve dogrusal uzay karsilastirmasi")
fig.colorbar(im, ax=ax[2], orientation="horizontal", fraction=0.15, pad=0.05)
plt.tight_layout()
plt.savefig(OUT / "07_gamma_ortalama.png", dpi=120)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(base_enc[0], err_tone[0], lw=1.4)
plt.xlabel("kodlu (sRGB) taban parlaklik")
plt.ylabel("hata (8 bit ton)")
plt.title("Hatanin parlaklikla degisimi")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "08_hata_egrisi.png", dpi=120)
plt.show()

## L1.3 Sonuç

1. Uç durumda (siyah + beyaz) naif ortalama 0.5000, doğru sonuç 0.7297 çıkmaktadır. 8 bit ölçekte 128'e karşı 186, yani **58 ton** fark vardır.
2. Hata her zaman aynı işaretlidir: naif sonuç daima olması gerekenden koyudur. $x^{1/\gamma}$ fonksiyonu konkav olduğundan, ortalamanın dönüşümü ile dönüşümlerin ortalaması arasındaki fark tek yönlüdür. Bu nedenle hata rastgele gürültü değil **sistematik yanlılıktır** ve üst üste uygulanan işlemlerde (örneğin Gauss piramidi) her seviyede birikir.
3. Aynı büyüklükteki kodlu fark için hata koyu bölgede açık bölgeye göre yaklaşık **3.6 kat** büyüktür. Nedeni $\frac{d}{dx}x^{1/\gamma} = \frac{1}{\gamma}x^{1/\gamma - 1}$ türevinin $x \to 0$ için sınırsız büyümesi, yani eğrinin koyu tarafta çok dik, açık tarafta neredeyse doğrusal olmasıdır.
4. Doğrusallaştırma, değer birleştiren tüm işlemler için gereklidir: yeniden boyutlandırma, alfa harmanlama, piramit inşası, çok kareli HDR ve gürültü giderme ortalamaları. Buna karşılık yalnızca sıralamaya dayanan işlemler (eşikleme, histogram germe, min ve maks) için gerekmez, çünkü gamma monoton artan bir dönüşümdür ve büyüklük sırasını korur.

---

# L1.4 - GSD ve şerit genişliği

## Problem

İğne deliği modelinde bir nesnenin görüntüdeki boyutu benzer üçgenlerden:

$$x = f_x \frac{X}{Z}$$

Odak uzaklığı katalogda milimetre cinsinden verilir; piksel cinsinden karşılığı için önce piksel boyutu bulunur:

$$s_{px} = \frac{W_s}{N_x}, \qquad f_x = \frac{f}{s_{px}}$$

Yer örnekleme aralığı (GSD), bir pikselin yeryüzünde karşılık geldiği mesafedir. Bağıntıda $x = 1$ piksel alınarak:

$$GSD = \frac{Z}{f_x}$$

Çözümlü Örnek 1.1 ile aynı kamera kullanılmıştır: $W_s = 13.2$ mm, $N_x = 5472$ px, $f = 8.8$ mm.

In [ ]:
W_S = 13.2
N_X = 5472
F_MM = 8.8

s_px = W_S / N_X
f_x = F_MM / s_px

print("piksel boyutu     s_px = W_s / N_x = %.1f / %d      = %.6e mm/px" % (W_S, N_X, s_px))
print("piksel cinsinden  f_x  = f / s_px  = %.1f / %.4e = %.1f px" % (F_MM, s_px, f_x))
print("birim kontrolu    mm / (mm/px) = px")

In [ ]:
heights = [60.0, 120.0, 240.0]

print("   Z (m)     GSD (m/px)    GSD (cm/px)    serit (m)    120 m'ye gore")
for Z in heights:
    gsd = Z / f_x
    swath = gsd * N_X
    tag = "referans" if Z == 120.0 else "%.2fx" % (Z / 120.0)
    print("   %6.1f    %.6f      %8.2f     %8.1f     %s" % (Z, gsd, gsd * 100, swath, tag))

### Çapraz doğrulama

Şerit genişliği ikinci bir yoldan, yatay görüş açısı üzerinden de hesaplanabilir:

$$FOV_x = 2\arctan\frac{W_s}{2f}, \qquad \text{serit} = 2Z\tan\frac{FOV_x}{2}$$

İki yolun aynı sonucu vermesi, hesap zincirinde birim veya cebir hatası bulunmadığını gösterir.

In [ ]:
fov = 2 * np.arctan(W_S / (2 * F_MM))
print("FOV_yatay = 2*arctan(%.1f / %.1f) = %.2f derece" % (W_S, 2 * F_MM, np.degrees(fov)))
print()
for Z in heights:
    swath_fov = 2 * Z * np.tan(fov / 2)
    swath_gsd = (Z / f_x) * N_X
    print("Z = %6.1f m  ->  FOV'dan %8.1f m | GSD'den %8.1f m | fark %.2e" % (
        Z, swath_fov, swath_gsd, abs(swath_fov - swath_gsd)))

In [ ]:
g60, g120, g240 = [Z / f_x for Z in heights]
print("GSD(120) / GSD(60)  = %.4f" % (g120 / g60))
print("GSD(240) / GSD(120) = %.4f" % (g240 / g120))
print()
print("kapali form: serit = Z * W_s / f = %.2f * Z" % (W_S / F_MM))

zz = np.linspace(20, 400, 300)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(zz, (zz / f_x) * 100, lw=1.6)
ax[0].scatter(heights, [(Z / f_x) * 100 for Z in heights], color="crimson", zorder=3)
for Z in heights:
    ax[0].annotate("%.2f cm/px" % ((Z / f_x) * 100), (Z, (Z / f_x) * 100),
                   textcoords="offset points", xytext=(6, -10), fontsize=9)
ax[0].set_xlabel("ucus yuksekligi Z (m)")
ax[0].set_ylabel("GSD (cm/px)")
ax[0].set_title("GSD = Z / f_x")
ax[0].grid(alpha=0.3)

ax[1].plot(zz, (zz / f_x) * N_X, lw=1.6, color="seagreen")
ax[1].scatter(heights, [(Z / f_x) * N_X for Z in heights], color="crimson", zorder=3)
for Z in heights:
    ax[1].annotate("%.0f m" % ((Z / f_x) * N_X), (Z, (Z / f_x) * N_X),
                   textcoords="offset points", xytext=(6, -10), fontsize=9)
ax[1].set_xlabel("ucus yuksekligi Z (m)")
ax[1].set_ylabel("serit genisligi (m)")
ax[1].set_title("serit = 1.5 * Z")
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "09_gsd.png", dpi=120)
plt.show()

## L1.4 Sonuç

| Z (m) | GSD (cm/px) | Şerit genişliği (m) | 120 m'ye göre |
|---|---|---|---|
| 60 | 1.64 | 90.0 | 0.50x |
| 120 | 3.29 | 180.0 | referans (Çözümlü Örnek 1.1) |
| 240 | 6.58 | 360.0 | 2.00x |

1. 120 m satırı Çözümlü Örnek 1.1 sonucuyla birebir örtüşmektedir (3.29 cm/px), bu da hesap zincirinin doğruluğunu gösterir.
2. $GSD = Z / f_x$ bağıntısında $f_x$ kameranın sabitidir; tek serbest değişken $Z$'dir. Bu nedenle GSD uçuş yüksekliğiyle **doğrusal** ölçeklenir. Ölçülen oranlar tam 2.0000 çıkmıştır.
3. Geometrik nedeni, görüş konisinin sabit açıyla açılmasıdır. Yükseklik iki katına çıktığında koni yeryüzünde iki kat geniş bir alanı tarar, ancak piksel sayısı değişmediğinden her piksele düşen yer alanı da iki katına çıkar.
4. Şerit genişliği bu kamera için $1.5 \cdot Z$ kapalı formuna indirgenir ve iki bağımsız yoldan (GSD ve FOV) aynı sonucu verir.
5. Tasarım ödünleşimi: yükseklik arttıkça tek geçişte taranan alan genişler, buna karşılık her pikselin yer çözünürlüğü kabalaşır. 10 cm GSD hedefi için gereken yükseklik $Z = GSD \cdot f_x = 0.10 \cdot 3648 \approx 365$ m'dir; bu da Çözümlü Örnek 1.1'in (d) şıkkıyla tutarlıdır.

---

# Genel değerlendirme

Dört görev, dijital kamera işlem hattının farklı adımlarındaki bilgi kaybını aynı yöntemle ele almıştır: kaybı denetimli biçimde üret, geri kazanmaya çalış, kalan farkı uygun bir metrikle ölç.

| Görev | Ölçülen büyüklük | Sonuç | Slayttaki iddia |
|---|---|---|---|
| L1.1 | PSNR (kanal bazlı) | G 31.22 dB, R 26.29 dB, B 26.58 dB | Renk bilgisinin üçte ikisi kestirimdir; hata R ve B'de yüksektir |
| L1.2 | SNR eğimi | 6.021 dB/bit, sapma sıfır | Her bit SNR'ı yaklaşık 6 dB artırır |
| L1.3 | Ton hatası | Uç durumda 58 ton; koyu bölgede 3.6 kat | Gamma kodlu veride ortalama almak yanlıştır |
| L1.4 | GSD | 1.64 / 3.29 / 6.58 cm/px | GSD, Z ile doğrusal ölçeklenir |

Ortak çıkarım, her adımda kaybedilen bilginin geri getirilemeyeceği ve sonraki tüm işlemlerin bu kayıpların üzerine kurulduğudur. Algoritma tasarımında hangi halkanın hata ürettiğini bilmek, hatayı sonradan gidermeye çalışmaktan daha belirleyicidir.